# Cloud Transfer Notebook

Copy public Yandex Disk files into your Google Drive. Run the cells below in order using a CPU runtime.

Files are streamed in 1 MiB chunks; a GPU is unnecessary. The original files remain on Yandex Disk.


## 1. Install dependencies


In [ ]:
%pip install -q 'requests>=2.32.4,<3' 'tqdm>=4.67,<5'


## 2. Connect Google Drive

Choose the Google account that should receive the files.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. Choose your files

Add your public links to `yandex_links`. Each dictionary key is the destination subdirectory name. For example:

```python
yandex_links = {
    "Documents": "https://disk.yandex.ru/d/YOUR_PUBLIC_KEY",
}
```

You can add multiple links, including links to files or nested folders. A folder's contents are copied into its named destination subdirectory. Keep `overwrite` disabled to preserve existing files; matching files are verified and skipped.


In [ ]:
destination = "/content/drive/MyDrive/cloud-import"
yandex_links = {}
overwrite = False


## 4. Load the transfer code

This notebook includes the transfer code and does not need any other project files.


In [ ]:
import hashlib
import logging
import os
import tempfile
import time
from dataclasses import dataclass
from pathlib import Path
from urllib.parse import unquote, urlsplit

import requests
from tqdm.auto import tqdm

API_URL = "https://cloud-api.yandex.net/v1/disk/public/resources"
CHUNK_SIZE = 1024 * 1024
PAGE_SIZE = 100
MAX_ATTEMPTS = 3
LOGGER = logging.getLogger(__name__)


class MigrationError(Exception):
    pass


class IntegrityError(MigrationError):
    pass


@dataclass
class MigrationResult:
    downloaded: int = 0
    skipped: int = 0
    failed: int = 0


def parse_public_url(url: str) -> tuple[str, str]:
    parsed = urlsplit(url)
    parts = parsed.path.strip("/").split("/")
    if (
        parsed.scheme != "https"
        or parsed.netloc not in {"disk.yandex.ru", "disk.yandex.com", "yadi.sk"}
        or len(parts) < 2
        or parts[0] not in {"d", "i"}
        or not parts[1]
    ):
        raise ValueError("Use a public HTTPS link from Yandex Disk (/d/... or /i/...).")
    public_key = f"https://{parsed.netloc}/{parts[0]}/{parts[1]}"
    path = unquote("/" + "/".join(parts[2:])) if len(parts) > 2 else ""
    return public_key, path


def safe_child(parent: Path, name: str, root: Path) -> Path:
    if (
        not isinstance(name, str)
        or not name
        or name in {".", ".."}
        or any(char in name for char in "/\\\0")
    ):
        raise MigrationError(f"Unsafe file or directory name: {name!r}")
    child = parent / name
    if child.is_symlink() or not child.resolve().is_relative_to(root):
        raise MigrationError(f"Destination escapes the selected directory: {name!r}")
    return child


def is_retryable(error: requests.RequestException, *, download: bool = False) -> bool:
    if isinstance(error, requests.HTTPError):
        status = error.response.status_code if error.response is not None else 0
        return (
            status == 429 or 500 <= status < 600 or (download and status in {401, 403})
        )
    return isinstance(
        error,
        (
            requests.ConnectionError,
            requests.Timeout,
            requests.exceptions.ChunkedEncodingError,
        ),
    )


def retry_delay(error: Exception, attempt: int) -> float:
    if isinstance(error, requests.HTTPError) and error.response is not None:
        value = error.response.headers.get("Retry-After", "")
        if value.isdecimal():
            return min(float(value), 60)
    return 2 ** (attempt - 1)


def request_error(error: requests.RequestException) -> str:
    if error.response is not None:
        return f"HTTP {error.response.status_code}"
    return type(error).__name__


def get_json(session: requests.Session, endpoint: str, params: dict) -> dict:
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            with session.get(endpoint, params=params, timeout=(10, 30)) as response:
                response.raise_for_status()
                data = response.json()
            if not isinstance(data, dict) or "error" in data:
                raise MigrationError("Yandex Disk API returned an invalid response.")
            return data
        except requests.RequestException as error:
            if attempt == MAX_ATTEMPTS or not is_retryable(error):
                raise MigrationError(
                    f"Yandex Disk API request failed: {request_error(error)}"
                ) from error
            time.sleep(retry_delay(error, attempt))
        except ValueError as error:
            raise MigrationError("Yandex Disk API returned invalid JSON.") from error
    raise AssertionError("Unreachable")


def get_resource(
    session: requests.Session, public_key: str, path: str, offset: int = 0
) -> dict:
    params = {
        "public_key": public_key,
        "limit": PAGE_SIZE,
        "offset": offset,
        "sort": "name",
    }
    if path:
        params["path"] = path
    return get_json(session, API_URL, params)


def iter_children(
    session: requests.Session, public_key: str, path: str, first_page: dict
):
    page = first_page
    offset = 0
    while True:
        embedded = page.get("_embedded")
        if not isinstance(embedded, dict):
            raise MigrationError(
                "Yandex Disk API returned an invalid directory listing."
            )
        items = embedded.get("items")
        total = embedded.get("total")
        if not isinstance(items, list) or not isinstance(total, int) or total < 0:
            raise MigrationError(
                "Yandex Disk API returned an invalid directory listing."
            )
        if not items and offset < total:
            raise MigrationError(
                "Directory listing ended before all entries were returned."
            )
        for item in items:
            if not isinstance(item, dict) or not isinstance(item.get("path"), str):
                raise MigrationError(
                    "Yandex Disk API returned an invalid directory entry."
                )
            yield item
        offset += len(items)
        if offset >= total:
            return
        page = get_resource(session, public_key, path, offset)


def file_metadata(resource: dict) -> tuple[int, str | None, str | None]:
    size = resource.get("size")
    if type(size) is not int or size < 0:
        raise MigrationError("File metadata is missing a valid size.")
    for algorithm in ("sha256", "md5"):
        expected = resource.get(algorithm)
        if expected is not None:
            length = 64 if algorithm == "sha256" else 32
            if (
                not isinstance(expected, str)
                or len(expected) != length
                or any(char not in "0123456789abcdefABCDEF" for char in expected)
            ):
                raise MigrationError(
                    f"File metadata has an invalid {algorithm} checksum."
                )
            return size, algorithm, expected.lower()
    return size, None, None


def verify_file(path: Path, resource: dict, *, require_checksum: bool = False) -> bool:
    size, algorithm, expected = file_metadata(resource)
    if path.stat().st_size != size:
        return False
    if algorithm is None:
        return not require_checksum
    digest = hashlib.new(algorithm, usedforsecurity=False)
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(CHUNK_SIZE), b""):
            digest.update(chunk)
    return digest.hexdigest() == expected


def download_file(
    session: requests.Session,
    public_key: str,
    resource: dict,
    target: Path,
    *,
    overwrite: bool,
) -> bool:
    """Return True for a new download and False for an already verified file."""
    expected_size, algorithm, expected_checksum = file_metadata(resource)
    if target.exists():
        if not target.is_file():
            raise MigrationError("Destination is not a regular file.")
        if not overwrite:
            if verify_file(target, resource, require_checksum=True):
                return False
            raise MigrationError(
                "Existing file differs or cannot be verified; set overwrite = True to replace it."
            )

    params = {"public_key": public_key, "path": resource["path"]}
    for attempt in range(1, MAX_ATTEMPTS + 1):
        temporary = None
        written = 0
        digest = hashlib.new(algorithm, usedforsecurity=False) if algorithm else None
        try:
            link = get_json(session, f"{API_URL}/download", params).get("href")
            if not isinstance(link, str) or urlsplit(link).scheme != "https":
                raise MigrationError(
                    "Yandex Disk did not return an HTTPS download link."
                )
            with session.get(link, stream=True, timeout=(10, 60)) as response:
                response.raise_for_status()
                with tempfile.NamedTemporaryFile(
                    dir=target.parent, prefix=".yadisk-", suffix=".part", delete=False
                ) as output:
                    temporary = Path(output.name)
                    with tqdm(
                        desc=target.name,
                        total=expected_size,
                        unit="B",
                        unit_scale=True,
                        unit_divisor=1024,
                        leave=False,
                    ) as progress:
                        for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                            if chunk:
                                output.write(chunk)
                                written += len(chunk)
                                if digest is not None:
                                    digest.update(chunk)
                                progress.update(len(chunk))
            if written != expected_size or (
                digest is not None and digest.hexdigest() != expected_checksum
            ):
                raise IntegrityError(
                    "Downloaded file does not match its size or checksum."
                )
            if target.is_symlink() or (target.exists() and not overwrite):
                raise MigrationError(
                    "Destination appeared during download; refusing to replace it."
                )
            os.replace(temporary, target)
            return True
        except requests.RequestException as error:
            if attempt == MAX_ATTEMPTS or not is_retryable(error, download=True):
                raise MigrationError(
                    f"Download failed: {request_error(error)}"
                ) from error
            LOGGER.warning(
                "Retrying %s (%s/%s)", target.name, attempt + 1, MAX_ATTEMPTS
            )
            time.sleep(retry_delay(error, attempt))
        except IntegrityError as error:
            if attempt == MAX_ATTEMPTS:
                raise
            LOGGER.warning("Retrying %s after integrity check failed", target.name)
            time.sleep(retry_delay(error, attempt))
        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)
    raise AssertionError("Unreachable")


def migrate(
    public_url: str,
    destination: str | Path,
    *,
    folder_name: str | None = None,
    overwrite: bool = False,
) -> MigrationResult:
    public_key, path = parse_public_url(public_url)
    root = Path(destination).expanduser().resolve()
    result = MigrationResult()
    with requests.Session() as session:
        session.headers.update({"User-Agent": "cloud-transfer-notebook/1.0"})
        resource = get_resource(session, public_key, path)
        root.mkdir(parents=True, exist_ok=True)
        if folder_name is not None:
            root = safe_child(root, folder_name, root)
            root.mkdir(exist_ok=True)
        pending = [(resource, root, path)]
        while pending:
            current, parent, current_path = pending.pop()
            try:
                resource_type = current.get("type")
                if resource_type == "file":
                    target = safe_child(parent, current["name"], root)
                    downloaded = download_file(
                        session, public_key, current, target, overwrite=overwrite
                    )
                    if downloaded:
                        result.downloaded += 1
                    else:
                        result.skipped += 1
                    LOGGER.info(
                        "%s: %s",
                        "Downloaded" if downloaded else "Verified, skipped",
                        target,
                    )
                elif resource_type == "dir":
                    if current is resource:
                        target = parent
                        first_page = current
                    else:
                        target = safe_child(parent, current["name"], root)
                        first_page = get_resource(session, public_key, current_path)
                    target.mkdir(exist_ok=True)
                    for child in iter_children(
                        session, public_key, current_path, first_page
                    ):
                        pending.append((child, target, child["path"]))
                else:
                    raise MigrationError(
                        f"Unsupported resource type: {resource_type!r}"
                    )
            except (MigrationError, OSError, KeyError) as error:
                result.failed += 1
                LOGGER.error("Failed %s: %s", current.get("name", "resource"), error)
    return result


## 5. Start copying

The report shows downloaded files, existing files verified and skipped, and errors. You can rerun this cell to retry failed transfers.


In [ ]:
if not Path("/content/drive/MyDrive").is_dir():
    raise ValueError("Connect Google Drive in step 2 before starting the transfer.")
if not yandex_links:
    raise ValueError("Add your public links to yandex_links in step 3.")
for folder_name, public_url in yandex_links.items():
    parse_public_url(public_url)
    safe_child(Path(destination).expanduser().resolve(), folder_name, Path(destination).expanduser().resolve())

LOGGER.setLevel(logging.INFO)
if not LOGGER.handlers:
    LOGGER.addHandler(logging.StreamHandler())
LOGGER.propagate = False
total = MigrationResult()
for folder_name, public_url in yandex_links.items():
    print(f"\nCopying into: {folder_name}")
    try:
        result = migrate(
            public_url, destination, folder_name=folder_name, overwrite=overwrite
        )
    except (MigrationError, OSError, ValueError) as error:
        LOGGER.error("Failed %s: %s", folder_name, error)
        result = MigrationResult(failed=1)
    total.downloaded += result.downloaded
    total.skipped += result.skipped
    total.failed += result.failed
    print(f"Downloaded: {result.downloaded}; verified, skipped: {result.skipped}; errors: {result.failed}")

print(f"\nTotal — downloaded: {total.downloaded}; verified, skipped: {total.skipped}; errors: {total.failed}")
